# Day 7 Lab 03: Storage Formats and Partitions

        **Layer:** Spark fundamentals  
        **Python reference:** `Lab_Files/labs/lab_03_storage_formats_and_partitions.py`

        This notebook is a sectioned companion version of the existing Python lab. It does not replace `run_lab.py` or the scripts under `Lab_Files/labs`.

        ## Learning Checkpoints
        - Add timestamp and date fields to raw events.
- Write partitioned Parquet by status.
- Read back partitioned lake files and inspect counts.

## 0. Notebook Setup

Run this first. It moves the kernel into `Lab_Files` and makes the shared lab helpers importable.

In [ ]:
from pathlib import Path
import os
import sys
import types
import typing

# PySpark 3.4 imports typing.io, which is absent in Python 3.14+.
if "typing.io" not in sys.modules:
    typing_io = types.ModuleType("typing.io")
    typing_io.IO = typing.IO
    typing_io.TextIO = typing.TextIO
    typing_io.BinaryIO = typing.BinaryIO
    sys.modules["typing.io"] = typing_io

def find_lab_files(start: Path) -> Path:
    relative_options = [
        Path("."),
        Path("Lab_Files"),
        Path("Day_07") / "Lab_Files",
        Path("Week_02") / "Day_07" / "Lab_Files",
    ]
    for root in [start, *start.parents]:
        for relative in relative_options:
            candidate = (root / relative).resolve()
            if (candidate / "labs" / "day7_common.py").exists():
                return candidate
    raise FileNotFoundError(
        "Could not find Week_02/Day_07/Lab_Files. "
        "Start Jupyter from the repository root, Week_02/Day_07, or Week_02/Day_07/Lab_Files."
    )

HERE = Path.cwd().resolve()
LAB_FILES = find_lab_files(HERE)

os.chdir(LAB_FILES)
labs_path = str(LAB_FILES / "labs")
if labs_path not in sys.path:
    sys.path.insert(0, labs_path)

print(f"Working directory: {Path.cwd()}")
print(f"Using lab helpers from: {labs_path}")


## 1. Import Lab Helpers

In [ ]:
from pyspark.sql import functions as F

import importlib
import day7_common
day7_common = importlib.reload(day7_common)


from day7_common import LAKE_DIR, OUTPUT_DIR, ORDER_SOURCE_FILES, ensure_output_dirs, read_order_events, require_source_data, spark_session, write_csv_dir

## 2. Start Spark

In [ ]:
require_source_data()
ensure_output_dirs()
spark = spark_session("Day7Notebook03StorageFormats")

## 3. Read and Type Event Time

In [ ]:
orders = (
    read_order_events(spark, [ORDER_SOURCE_FILES[0]])
    .withColumn("event_time_ts", F.to_timestamp("event_time"))
    .withColumn("event_date", F.to_date("event_time_ts"))
)
orders.select("event_id", "status", "event_time", "event_time_ts", "event_date").show(10, truncate=False)

## 4. Write Partitioned Parquet

In [ ]:
parquet_path = LAKE_DIR / "playground" / "orders_by_status_parquet"
orders.write.mode("overwrite").partitionBy("status").parquet(str(parquet_path))
print(f"Wrote partitioned Parquet to {parquet_path}")

## 5. Read Back and Count Partitions

In [ ]:
partition_counts = spark.read.parquet(str(parquet_path)).groupBy("status").count().orderBy("status")
partition_counts.show(truncate=False)

## 6. Write Learner Output

In [ ]:
write_csv_dir(partition_counts, OUTPUT_DIR / "notebook_03_partition_counts")

## Clean Shutdown

Stop the SparkSession when you are done with the notebook. The next notebook will create its own session.

In [ ]:
# Run this at the end of the notebook, or before restarting/rerunning the lab.
spark.stop()